In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os


In [20]:
data_path = os.path.join('..',"data",'raw')

customers = pd.read_csv(f'{data_path}/customers.csv')
sellers = pd.read_csv(f'{data_path}/sellers.csv')
orders = pd.read_csv(f'{data_path}/orders.csv')
order_items = pd.read_csv(f'{data_path}/order_items.csv')
geolocation = pd.read_csv(f'{data_path}/geolocation.csv')

In [21]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05


In [22]:
distance = orders[['order_id','customer_id','order_delivered_customer_date','order_purchase_timestamp']]

In [23]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [24]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [25]:
# select on records which order is delivered successfully, and not null also.
order_delivered = orders[~orders['order_delivered_customer_date'].isna()]

In [26]:
order_delivered.info()

<class 'pandas.DataFrame'>
Index: 96476 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       96476 non-null  str  
 1   customer_id                    96476 non-null  str  
 2   order_status                   96476 non-null  str  
 3   order_purchase_timestamp       96476 non-null  str  
 4   order_approved_at              96462 non-null  str  
 5   order_delivered_carrier_date   96475 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  96476 non-null  str  
dtypes: str(8)
memory usage: 6.6 MB


In [27]:
order_delivered['order_status'].value_counts(dropna=False)

order_status
delivered    96470
canceled         6
Name: count, dtype: int64

In [28]:
#select important features only.
order_delivered = order_delivered[['order_id','customer_id','order_purchase_timestamp','order_delivered_customer_date']]

In [31]:
# get seller id from order_item table
delivered_orders_with_seller =  order_delivered.merge(on='order_id'
                                                      ,how='left',
                                                      right = order_items[['order_id',  'seller_id']],
                                                      validate="one_to_one")

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
                         order_id
0008288aa423d2a3f00fcb17cd7d8719
00143d0f86d6fbd9f9b38ab440ac16f5
00143d0f86d6fbd9f9b38ab440ac16f5
001ab0a7578dd66cd4b0a71f5b6e1e41
001ab0a7578dd66cd4b0a71f5b6e1e41 ...

In [12]:
# add customer prefix zip code
delivered_orders_sellers_customers_zip  = delivered_orders_with_seller.merge(customers[['customer_id','customer_zip_code_prefix']],
                                                                             how='left',
                                                                             on='customer_id')

In [13]:
# add seller prefix zip code
delivered_orders_sellers_customers_zip =delivered_orders_sellers_customers_zip.merge(right = sellers[['seller_id','seller_zip_code_prefix']],on ='seller_id',how='left')

In [14]:
cust_cordinate = delivered_orders_sellers_customers_zip.merge(right=geolocation[['geolocation_zip_code_prefix','geolocation_lat','geolocation_lng']],
                                             left_on='customer_zip_code_prefix',
                                             right_on ='geolocation_zip_code_prefix',how='left')


In [15]:
cust_cordinate.info()

<class 'pandas.DataFrame'>
RangeIndex: 11222953 entries, 0 to 11222952
Data columns (total 10 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   order_id                       str    
 1   customer_id                    str    
 2   order_purchase_timestamp       str    
 3   order_delivered_customer_date  str    
 4   seller_id                      str    
 5   customer_zip_code_prefix       int64  
 6   seller_zip_code_prefix         int64  
 7   geolocation_zip_code_prefix    float64
 8   geolocation_lat                float64
 9   geolocation_lng                float64
dtypes: float64(3), int64(2), str(5)
memory usage: 856.2 MB


In [16]:
cust_cordinate.order_id.is_unique

False

In [17]:
order_delivered.order_id.is_unique

True

In [18]:
cust_cordinate.duplicated().count()

np.int64(11222953)

In [19]:
order_delivered.duplicated().count()

np.int64(96476)

In [ ]:
order_delivered.merge